In [1]:
# ============================================================
# SETUP
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df_sales = pd.read_csv('data/processed/cleaned_retail_sales.csv')
df_sales['Order_Date'] = pd.to_datetime(df_sales['Order_Date'])
rfm = pd.read_csv('data/processed/rfm_analysis.csv')
cust = pd.read_csv('data/processed/customer_clv.csv')

print(df_sales.shape, rfm.shape, cust.shape)

(10000, 30) (1986, 10) (1986, 10)


In [2]:
# ============================================================
# CORE BUSINESS KPIS
# ============================================================

kpis = {}
kpis['Total_Revenue'] = df_sales['Sales_Capped'].sum()
kpis['Total_Orders'] = df_sales['Order_ID'].nunique()
kpis['Total_Customers'] = df_sales['Customer_ID'].nunique()
kpis['Avg_Order_Value'] = df_sales['Sales_Capped'].mean()
kpis['Total_Profit'] = df_sales['Profit_Capped'].sum()
kpis['Profit_Margin_Pct'] = (df_sales['Profit_Capped'].sum() / df_sales['Sales_Capped'].sum()) * 100
kpis['Repeat_Customer_Rate'] = (df_sales.groupby('Customer_ID')['Order_ID'].count() > 1).mean() * 100
kpis['Churn_Rate_Pct'] = (rfm['Recency'] > 180).mean() * 100

for name, value in kpis.items():
    print(f"{name}: {value:,.2f}")

Total_Revenue: 1,078,670.98
Total_Orders: 10,000.00
Total_Customers: 1,986.00
Avg_Order_Value: 107.87
Total_Profit: 198,274.85
Profit_Margin_Pct: 18.38
Repeat_Customer_Rate: 96.27
Churn_Rate_Pct: 10.88


In [3]:
# ============================================================
# TIME-BASED & SEGMENT-LEVEL KPIS
# ============================================================

df_sales['Order_Month'] = df_sales['Order_Date'].dt.to_period('M')
monthly_kpis = df_sales.groupby('Order_Month').agg(
    Revenue=('Sales_Capped', 'sum'),
    Orders=('Order_ID', 'nunique'),
    Avg_Order_Value=('Sales_Capped', 'mean')
).reset_index()

monthly_kpis['Revenue_MoM_Growth_Pct'] = monthly_kpis['Revenue'].pct_change() * 100

category_kpis = df_sales.groupby('Product_Category', observed=True).agg(
    Revenue=('Sales_Capped', 'sum'),
    Orders=('Order_ID', 'nunique'),
    Avg_Order_Value=('Sales_Capped', 'mean'),
    Total_Profit=('Profit_Capped', 'sum')
).reset_index().sort_values('Revenue', ascending=False)

regional_kpis = df_sales.groupby('Region', observed=True).agg(
    Revenue=('Sales_Capped', 'sum'),
    Orders=('Order_ID', 'nunique'),
    Customers=('Customer_ID', 'nunique')
).reset_index().sort_values('Revenue', ascending=False)

print("MONTHLY (last 5 rows):")
print(monthly_kpis.tail())
print()
print("BY CATEGORY:")
print(category_kpis)
print()
print("BY REGION:")
print(regional_kpis)

MONTHLY (last 5 rows):
   Order_Month       Revenue  Orders  Avg_Order_Value  Revenue_MoM_Growth_Pct
9      2022-10  77173.028698     744       103.727189                1.013698
10     2022-11  78702.980474     720       109.309695                1.982495
11     2022-12  82235.806623     744       110.531998                4.488809
12     2023-01  78408.906190     744       105.388315               -4.653570
13     2023-02  54090.000133     496       109.052420              -31.015489

BY CATEGORY:
  Product_Category        Revenue  Orders  Avg_Order_Value  Total_Profit
1      Electronics  328422.713112    3029       108.426119  62146.581248
3  Office Supplies  321280.864068    2949       108.945698  57981.790645
2        Furniture  214958.032671    2011       106.891115  39702.384657
0         Clothing  214009.369624    2011       106.419378  38444.098098

BY REGION:
    Region        Revenue  Orders  Customers
1     East  318581.371468    2968       1553
0  Central  275688.179783   

In [4]:
# ============================================================
# EXCLUDING THE PARTIAL FINAL MONTH
# ============================================================

monthly_kpis_complete = monthly_kpis[monthly_kpis['Order_Month'] < pd.Period('2023-02', freq='M')].copy()
monthly_kpis_complete['Revenue_MoM_Growth_Pct'] = monthly_kpis_complete['Revenue'].pct_change() * 100

print(monthly_kpis_complete.tail())
print()
print("Avg MoM growth (complete months only):", monthly_kpis_complete['Revenue_MoM_Growth_Pct'].mean().round(2), "%")

   Order_Month       Revenue  Orders  Avg_Order_Value  Revenue_MoM_Growth_Pct
8      2022-09  76398.577746     720       106.109136               -7.049308
9      2022-10  77173.028698     744       103.727189                1.013698
10     2022-11  78702.980474     720       109.309695                1.982495
11     2022-12  82235.806623     744       110.531998                4.488809
12     2023-01  78408.906190     744       105.388315               -4.653570

Avg MoM growth (complete months only): -0.23 %


In [5]:
# ============================================================
# EXECUTIVE SUMMARY
# ============================================================

exec_summary = f"""
RETAIL & MARKETING ANALYTICS — EXECUTIVE SUMMARY
==================================================

BUSINESS OVERVIEW
------------------
Total Revenue: ${kpis['Total_Revenue']:,.2f}
Total Orders: {kpis['Total_Orders']:,.0f}
Total Customers: {kpis['Total_Customers']:,.0f}
Average Order Value: ${kpis['Avg_Order_Value']:,.2f}
Profit Margin: {kpis['Profit_Margin_Pct']:.1f}%
Churn Rate (180+ days inactive): {kpis['Churn_Rate_Pct']:.1f}%

KEY FINDINGS
------------------
1. Revenue is concentrated in two categories: Electronics (${category_kpis.iloc[0]['Revenue']:,.0f}) and
   Office Supplies (${category_kpis.iloc[1]['Revenue']:,.0f}) together drive {(category_kpis.iloc[0]['Revenue']+category_kpis.iloc[1]['Revenue'])/kpis['Total_Revenue']*100:.0f}% of revenue.

2. Monthly sales are stable (avg MoM growth: -0.2%) with no dominant seasonal pattern --
   this is a steady-state business, not currently in a growth or decline phase.

3. Customer segmentation (RFM) identifies 278 "At Risk" customers (14.0% of base) who carry
   historically strong value (avg. $698 spend) but have not ordered in 126+ days on average --
   the single highest-priority group for retention intervention.

4. K-Means clustering independently confirms 3 natural customer tiers: Lost/Dormant (18.8%),
   Core (48.2%), and High-Value (32.9%) -- validating that RFM's business segments reflect real,
   statistically distinct behavioral groups, not arbitrary thresholds.

5. Regional performance is uneven: South underperforms East by 33% in revenue, driven by both
   fewer customers and lower spend per customer -- a compounding gap worth prioritizing.

6. Discount shows no measurable correlation with sales or profit (~0.00-0.02) -- current
   discounting practice is not demonstrably driving incremental revenue.

RECOMMENDATIONS
------------------
Immediate (0-30 days):
- Launch a win-back campaign targeted at the 278 "At Risk" customers (~$194K in historical
  revenue exposure)

Short-term (2-3 months):
- Investigate South region's lower customer acquisition and spend-per-customer; pilot a
  region-specific promotion

Long-term (6-12 months):
- Reassess discount strategy given the lack of correlation with sales/profit; consider testing
  whether discounts are being applied where they add the least value
"""

print(exec_summary)

with open('outputs/reports/executive_summary.txt', 'w') as f:
    f.write(exec_summary)

with open('outputs/reports/monthly_kpis.csv', 'w') as f:
    monthly_kpis_complete.to_csv(f, index=False)
category_kpis.to_csv('outputs/reports/category_kpis.csv', index=False)
regional_kpis.to_csv('outputs/reports/regional_kpis.csv', index=False)

print("\nAll reports saved to outputs/reports/")


RETAIL & MARKETING ANALYTICS — EXECUTIVE SUMMARY

BUSINESS OVERVIEW
------------------
Total Revenue: $1,078,670.98
Total Orders: 10,000
Total Customers: 1,986
Average Order Value: $107.87
Profit Margin: 18.4%
Churn Rate (180+ days inactive): 10.9%

KEY FINDINGS
------------------
1. Revenue is concentrated in two categories: Electronics ($328,423) and
   Office Supplies ($321,281) together drive 60% of revenue.

2. Monthly sales are stable (avg MoM growth: -0.2%) with no dominant seasonal pattern --
   this is a steady-state business, not currently in a growth or decline phase.

3. Customer segmentation (RFM) identifies 278 "At Risk" customers (14.0% of base) who carry
   historically strong value (avg. $698 spend) but have not ordered in 126+ days on average --
   the single highest-priority group for retention intervention.

4. K-Means clustering independently confirms 3 natural customer tiers: Lost/Dormant (18.8%),
   Core (48.2%), and High-Value (32.9%) -- validating that RFM's b